# 03 — Built-in Tools Tour

**Goal of this episode:** prove that `create_deep_agent` hands you a full toolbox for free —
we won't write a single custom tool. We'll give the agent a small sandbox folder, ask it
to do five jobs, and then print a scorecard showing which built-in tools it actually used.

**What we'll code (5 steps):**

1. Imports + pick a model
2. Seed a throwaway workspace with sample files
3. Build the agent (backend + middleware + system prompt)
4. Write prompts that each target specific tools, plus a helper to read tool calls
5. Run the conversation and print a tool-coverage scorecard

## The 10 built-in tools (and where each one comes from)

| Tool | What it does | Who provides it |
| --- | --- | --- |
| `ls` | list a directory | filesystem middleware (always on) |
| `read_file` | read a file | filesystem middleware |
| `write_file` | create / overwrite a file | filesystem middleware |
| `edit_file` | find-and-replace inside a file | filesystem middleware |
| `glob` | find files by pattern (`**/*.txt`) | filesystem middleware |
| `grep` | search file contents | filesystem middleware |
| `delete` | remove a file | filesystem middleware |
| `execute` | run a shell command | **`LocalShellBackend`** only |
| `task` | hand work to a subagent | subagent middleware (always on) |
| `write_todos` | keep a to-do plan | **`TodoListMiddleware`** (opt-in) |

> 🎬 **Talking point:** the *backend* decides **where** the file tools operate and whether
> `execute` exists; *middleware* adds extra abilities like planning.

## How to follow along

- Run the cells top to bottom — each step defines something the next step uses.
- Steps 1–4 only **define** things (no model calls yet). Step 5 is where the agent runs.
- Prefer a script? The companion `03-builtin_tools.py` has the same code.
- Need Ollama running locally (`ollama serve`) for the default model.

### Step 1: Imports and model configuration

**What we're coding:** the imports, the model, the sandbox path, and a checklist of tools.

- `create_deep_agent` — the one function that builds the whole agent.
- `LocalShellBackend` — real disk **plus** the `execute` shell tool.
- `TodoListMiddleware` — opts us in to `write_todos`.
- `BUILT_INS` — the list we'll check off at the end.

> 🎬 **Say:** "Swap the model by changing one line — cloud or local, the rest of the code doesn't care."

In [ ]:
# %% Step 1: Imports and model configuration
import os  # read OLLAMA_BASE_URL from the environment
from pathlib import Path  # build the sandbox folder path

from deepagents import create_deep_agent  # builds the agent + its built-in tools
from deepagents.backends import LocalShellBackend  # real disk + the `execute` shell tool
from dotenv import load_dotenv  # load API keys from a .env file
from langchain.agents.middleware import TodoListMiddleware  # adds the `write_todos` tool
from langchain_ollama import ChatOllama  # local model via Ollama
from rich import print  # colored console output ([bold], [green], ...)

################################ Environment & Model Configuration ################################

# Load .env first so keys like OPENAI_API_KEY are available below
load_dotenv()

# Pick ONE model. The LAST assignment to MODEL wins.
# Option A — cloud: a provider string is all create_deep_agent needs
MODEL = "openai:gpt-4.1-nano"  # gpt-4.1-nano
# Option B — local: a model instance lets us set base_url and temperature
OLLANMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
# Other small models to try: llama3.2 granite4:1b qwen3.5:2b
MODEL = ChatOllama(model="llama3.2", base_url=OLLANMA_BASE_URL, temperature=0)  # 0 = repeatable

# The sandbox folder: every file and shell tool call is locked inside it.
# Scripts have __file__; notebooks don't, so fall back to the current folder.
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
WORKSPACE = HERE / "workspace"

# The 10 built-in tools — our scorecard checklist for Step 5
BUILT_INS = (
    "write_todos",  # planning (TodoListMiddleware)
    "ls",  # filesystem tools ...
    "read_file",
    "write_file",
    "edit_file",
    "glob",
    "grep",
    "execute",  # shell (LocalShellBackend)
    "task",  # subagent delegation
    "delete",
)
# print(MODEL)

### Step 2: Seed the workspace with sample files

**What we're coding:** a function that resets the sandbox to a known state.

```
workspace/
├── notes.md          "# Notes / - status: draft"   ← agent will edit this
└── data/
    ├── fruits.txt    apple, banana, cherry          ← agent will read this
    ├── colors.txt    red, blue, green               ← agent will grep this
    └── veggies.txt   (removed)                      ← agent will create, then delete this
```

> 🎬 **Say:** "Every run starts clean, so the demo behaves the same every time we re-run it."

In [ ]:
# %% Step 2: Create safe, repeatable data for the agent to explore
def seed_workspace() -> None:
    """Reset only this tutorial's sample files before each run."""
    # 1) Make workspace/data/ (no error if it already exists)
    data = WORKSPACE / "data"
    data.mkdir(parents=True, exist_ok=True)
    # 2) Files the agent will read (read_file) and search (grep)
    (data / "fruits.txt").write_text("apple\nbanana\ncherry\n")
    (data / "colors.txt").write_text("red\nblue\ngreen\n")
    # 3) A file the agent will change "draft" -> "final" in (edit_file)
    (WORKSPACE / "notes.md").write_text("# Notes\n- status: draft\n")
    # 4) Remove veggies.txt so the agent has to create it (write_file)
    (data / "veggies.txt").unlink(missing_ok=True)

### Step 3: Build the agent

**What we're coding:** `create_deep_agent(model, backend, middleware, system_prompt)`.

| Argument | Why it's here |
| --- | --- |
| `backend=LocalShellBackend(...)` | file tools hit real disk inside `WORKSPACE`, and `execute` is added |
| `middleware=[TodoListMiddleware()]` | adds `write_todos` |
| `system_prompt=...` | tells the model to use the exact tool we name (not shell for everything) |

> 🎬 **Say:** "Notice we pass **no** `tools=` list. Everything this agent can do comes built in."

⚠️ `LocalShellBackend` runs real commands on your machine. `root_dir` keeps file paths
inside the workspace and `timeout` stops runaway commands — but it is not a security sandbox.

In [ ]:
# %% Step 3: Build an agent with every available harness tool
def build_agent():
    """LocalShellBackend adds execute; middleware adds write_todos."""
    # Backend = WHERE tools operate. Root everything at WORKSPACE; kill commands after 30s.
    backend = LocalShellBackend(root_dir=str(WORKSPACE.resolve()), timeout=30)
    return create_deep_agent(
        model=MODEL,  # the model from Step 1
        backend=backend,  # gives us the file tools + execute
        middleware=[TodoListMiddleware()],  # opt in to the write_todos planning tool
        # No tools= argument: every tool comes built in.
        # The prompt pushes the model to call the exact tool we name, so each one gets exercised.
        system_prompt="""You are demonstrating Deep Agents tools in a throwaway workspace.
            For every requested operation, call the named tool exactly; never replace a
            filesystem operation with execute. Use relative paths. Keep final answers brief.""",
    )

### Step 4: Prompts that target every tool + a helper to read tool calls

**What we're coding:**

1. `PROMPTS` — five user messages. Each one names the tools it should trigger:

   | Turn | Tools we expect |
   | --- | --- |
   | 1 | `write_todos`, `ls`, `read_file` |
   | 2 | `write_file`, `edit_file` |
   | 3 | `glob`, `grep`, `execute` |
   | 4 | `task`, `delete`, `write_todos` |
   | 5 | `write_file`, `glob`/`ls`, `delete` |

2. `tool_names(messages)` — scans the agent's messages and returns the names of the tools it called.

> 🎬 **Say:** "Every tool call the model makes is saved on an `AIMessage` in `.tool_calls` —
> so we can check what it *actually* did, not just what it *says* it did."

In [ ]:
# %% Step 4: Short prompts that explicitly cover every tool
# Five turns. Naming the tool in each prompt nudges the model to call it.
PROMPTS = [
    (  # Turn 1 — tools: write_todos, ls, read_file
        "1. Use write_todos to plan these two items, then use ls on data and "
        "read_file on data/fruits.txt and tell me what fruits are listed."
    ),
    (  # Turn 2 — tools: write_file, edit_file
        "2. Use write_file to create data/veggies.txt containing carrot, pepper, and basil on separate lines."
        "Then use edit_file to change 'draft' to 'final' in notes.md."
    ),
    (  # Turn 3 — tools: glob, grep, execute
        "3. Use glob to find every .txt file. Use grep to find the file containing 'blue'. "
        "Use execute to run exactly: wc -l data/*.txt"
    ),
    (  # Turn 4 — tools: task (subagent), delete, write_todos
        "4. Use task to delegate counting all lines in data/*.txt to a subagent."
        "Then use delete to remove data/veggies.txt, complete the todos, and summarize the results."
    ),
    (  # Turn 5 — open-ended: write_file (table into notes.md), then find + delete colors.txt
        "5. List all of the built-in tools available for me to use with deepagents and a brief description of each."
        "Write the tools and description to notes.md file as simple table"
        "find and delete data/colors.txt file from the workspace."
    ),
]


def tool_names(messages: list) -> set[str]:
    """Collect tool names from the agent's LangChain messages."""
    tool_names = set()  # a set, so each tool is counted once
    for message in messages:
        # Only AIMessages have .tool_calls; for others getattr returns [] and the loop skips
        for call in getattr(message, "tool_calls", []) or []:
            # Each call is a dict like {"name": "ls", "args": {...}, "id": "..."}
            tool_names.add(call["name"])
    return tool_names

### Step 5: Run the conversation and print the scorecard

**What we're coding:** `main()` — the loop that drives everything.

1. `seed_workspace()` → fresh files.
2. `build_agent()` → the agent from Step 3.
3. For each prompt: `agent.invoke({"messages": history + [new prompt]})`.
4. Keep `result["messages"]` as the new history, so the agent remembers its to-do list across turns.
5. Pull tool names out with `tool_names()` and add them to the `called` set.
6. Print ✅/❌ for every tool in `BUILT_INS`.

> 🎬 **Say:** "`invoke` returns the full state — `messages` holds the entire conversation, including
> every tool call and tool result. The last message is the agent's final answer."

💡 Small local models sometimes skip a tool or call the wrong one — a red **no** is a useful
result, not a bug. Try a bigger model (or the OpenAI line in Step 1) and compare scorecards.

In [ ]:
# %% Step 5: Run the guided conversation and report actual coverage
def main() -> None:
    """Invoke each prompt, preserving messages so the agent can continue its plan."""
    seed_workspace()  # start from known files every run
    # agent = Step 3; messages = conversation history; called = tools seen so far
    agent, messages, called = build_agent(), [], set()
    print(f"[dim]model: {MODEL}  workspace: {WORKSPACE}[/dim]")

    for turn, prompt in enumerate(PROMPTS, start=1):
        print(f"\n[bold magenta]--- Turn {turn} ---[/bold magenta]")
        # Send the WHOLE history + the new prompt (no checkpointer, so we carry memory ourselves)
        result = agent.invoke({"messages": messages + [{"role": "user", "content": prompt}]})
        print("Got results finding new tool calls...")
        # result is the final state dict; its "messages" is the updated history
        messages = result["messages"]
        # Tool names across the whole history so far (cumulative, not just this turn)
        tools_used = tool_names(messages)
        # Set difference = tools seen for the first time on this turn
        new_calls = tools_used - called

        called.update(new_calls)  # add them to the running total
        print("tools used:", ", ".join(sorted(tools_used)) or "none")
        print("  answer: \n")
        print(messages[-1])  # last message = the agent's reply for this turn

    # Scorecard: did the agent call each built-in tool at least once?
    print("\n[bold cyan]Built-in tool coverage[/bold cyan]")
    for name in BUILT_INS:
        marker = "[green]yes[/green]" if name in called else "[red]no[/red]"
        print(f"  {name:11} {marker}")  # :11 pads names so the yes/no column lines up
    # .text pulls plain text out of the message (works for Ollama and OpenAI formats)
    print(f"\n[bold green]Final answer:[/bold green] {messages[-1].text}")


# Runs when executed as a script AND when this notebook cell runs (__name__ == "__main__" in Jupyter)
if __name__ == "__main__":
    main()